# Simple RNN — Next Word Prediction

Sentence: **"the quick brown fox jumps over the lazy dog"**

Task: show RNN a word, it guesses the next word.

Whole idea in two lines:

```
h = tanh(x @ Wx + h @ Wh + bh)   # memory: what I read now + what I remember
y = h @ Wy + by                  # guess: score for every word in vocab
```

`h` is the memory. It starts empty and gets updated after every word.
Same `Wx`, `Wh`, `Wy` used at every step — that is why it is called *recurrent*.

No classes, no hidden helpers below. Just tensors and those two lines.


## 1. Words to numbers

Computer cannot read words, only numbers. So give each unique word an ID.


In [1]:
import torch

torch.manual_seed(0)

sentence = "the quick brown fox jumps over the lazy dog"
words = sentence.split()

vocab = sorted(set(words))                       # unique words, fixed order
word2id = {w: i for i, w in enumerate(vocab)}    # word -> number
id2word = {i: w for w, i in word2id.items()}     # number -> word

print("words  :", words)
print("vocab  :", vocab)
print("word2id:", word2id)


words  : ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
vocab  : ['brown', 'dog', 'fox', 'jumps', 'lazy', 'over', 'quick', 'the']
word2id: {'brown': 0, 'dog': 1, 'fox': 2, 'jumps': 3, 'lazy': 4, 'over': 5, 'quick': 6, 'the': 7}


## 2. Training pairs

We want "given this word, what comes next".
So shift the sentence by one: input is every word except last, target is every word except first.


In [2]:
ids = [word2id[w] for w in words]

inputs  = ids[:-1]   # the quick brown fox jumps over the lazy
targets = ids[1:]    #     quick brown fox jumps over the lazy dog

for a, b in zip(inputs, targets):
    print(f"{id2word[a]:>6}  ->  {id2word[b]}")


   the  ->  quick
 quick  ->  brown
 brown  ->  fox
   fox  ->  jumps
 jumps  ->  over
  over  ->  the
   the  ->  lazy
  lazy  ->  dog


## 3. One-hot the input word

A word ID like `7` is meaningless as a number (word 7 is not "bigger" than word 3).
So turn it into a vector of zeros with a single `1` at that position.

`the` (id 7) with vocab size 8 becomes `[0,0,0,0,0,0,0,1]`.


In [3]:
vocab_size = len(vocab)

def one_hot(word_id):
    v = torch.zeros(vocab_size)
    v[word_id] = 1.0
    return v

print("vocab_size:", vocab_size)
print("the ->", one_hot(word2id["the"]))
print("fox ->", one_hot(word2id["fox"]))


vocab_size: 8
the -> tensor([0., 0., 0., 0., 0., 0., 0., 1.])
fox -> tensor([0., 0., 1., 0., 0., 0., 0., 0.])


## 4. The weights

Three matrices, that is the entire RNN.

| name | shape | job |
|---|---|---|
| `Wx` | vocab_size x hidden_size | read the current word into memory |
| `Wh` | hidden_size x hidden_size | carry old memory forward |
| `Wy` | hidden_size x vocab_size | turn memory into a score per word |

Plus two bias vectors `bh`, `by`. `requires_grad=True` tells PyTorch to track these
so it can compute gradients later.


In [5]:
hidden_size = 6   # size of the memory vector. bigger = more room to remember

Wx = torch.randn(vocab_size,  hidden_size) * 0.3
Wh = torch.randn(hidden_size, hidden_size) * 0.3
Wy = torch.randn(hidden_size, vocab_size)  * 0.3
bh = torch.zeros(hidden_size)
by = torch.zeros(vocab_size)

params = [Wx, Wh, Wy, bh, by]
for p in params:
    p.requires_grad = True

for name, p in zip(["Wx", "Wh", "Wy", "bh", "by"], params):
    print(f"{name}: {tuple(p.shape)}")


Wx: (8, 6)
Wh: (6, 6)
Wy: (6, 8)
bh: (6,)
by: (8,)


## 5. One step

Take current word + old memory, produce new memory + scores.

```
h_new  = tanh(x @ Wx + h_old @ Wh + bh)
scores = h_new @ Wy + by
```

Why `tanh`? It squashes everything into -1..1, so memory cannot blow up to huge numbers
after many steps.

`scores` has one number per vocab word — higher means "more likely next".


In [9]:
def step(x, h):
    h = torch.tanh(x @ Wx + h @ Wh + bh)   # new memory
    scores = h @ Wy + by                   # one score per vocab word
    return h, scores


# try it once, untrained: feed "the" with empty memory
h0 = torch.zeros(hidden_size)
h1, scores = step(one_hot(word2id["the"]), h0)

print("memory after 'the':", h1.detach())
print("best guess        :", id2word[scores.argmax().item()], "(random, not trained yet)")


memory after 'the': tensor([-0.8374, -0.4583,  0.8336,  0.2563,  0.3538,  0.8354])
best guess        : quick (random, not trained yet)


## 6. Run over the whole sentence

Loop `step` across all words. Memory `h` starts at zeros and is passed along,
so word 5 sees the effect of words 1-4.


In [10]:
def run(word_ids):
    h = torch.zeros(hidden_size)   # empty memory
    all_scores = []

    for word_id in word_ids:
        h, scores = step(one_hot(word_id), h)
        all_scores.append(scores)

    return torch.stack(all_scores)   # shape: (num_words, vocab_size)


print("output shape:", run(inputs).shape, " -> one row of scores per input word")


output shape: torch.Size([8, 8])  -> one row of scores per input word


## 7. Train

Loop many times:

1. `run(inputs)` — forward pass, get scores at every step
2. `cross_entropy` — how wrong the scores are vs the correct next word
3. `loss.backward()` — gradients flow back through *every* step of the loop.
   This is **BPTT** (backpropagation through time)
4. nudge each weight a little in the direction that lowers loss

Doing the update by hand (`p -= lr * p.grad`) instead of an optimizer, so nothing is hidden.
Loss should drop toward 0.


In [11]:
target_tensor = torch.tensor(targets)
lr = 0.1

for epoch in range(300):
    scores = run(inputs)                                        # 1. forward
    loss = torch.nn.functional.cross_entropy(scores, target_tensor)  # 2. how wrong

    loss.backward()                                             # 3. BPTT

    with torch.no_grad():                                       # 4. update
        for p in params:
            p -= lr * p.grad
            p.grad.zero_()          # clear, else gradients pile up

    if epoch % 30 == 0 or epoch == 299:
        print(f"epoch {epoch:>3}  loss {loss.item():.4f}")


epoch   0  loss 0.1424
epoch  30  loss 0.1226
epoch  60  loss 0.1072
epoch  90  loss 0.0949
epoch 120  loss 0.0849
epoch 150  loss 0.0766
epoch 180  loss 0.0698
epoch 210  loss 0.0639
epoch 240  loss 0.0589
epoch 270  loss 0.0546
epoch 299  loss 0.0510


## 8. Did it learn?

Feed the sentence again, take the highest-scoring word at each step, compare to truth.


In [12]:
with torch.no_grad():
    predicted = run(inputs).argmax(dim=1)

print(f"{'input':>6}  {'predicted':>10}  {'correct':>8}   ok?")
for i in range(len(inputs)):
    inp  = id2word[inputs[i]]
    pred = id2word[predicted[i].item()]
    true = id2word[targets[i]]
    print(f"{inp:>6}  {pred:>10}  {true:>8}   {'yes' if pred == true else 'NO'}")


 input   predicted   correct   ok?
   the       quick     quick   yes
 quick       brown     brown   yes
 brown         fox       fox   yes
   fox       jumps     jumps   yes
 jumps        over      over   yes
  over         the       the   yes
   the        lazy      lazy   yes
  lazy         dog       dog   yes


## 9. Generate

Give only the first word, then feed each prediction back in as the next input.

`the` appears twice in the sentence — before `quick` and before `lazy`.
Same input word, different correct answer. Only way to tell them apart is the memory `h`.
If generation gets both right, the memory is really doing work.


In [13]:
def generate(start_word, n_words):
    result = [start_word]
    h = torch.zeros(hidden_size)
    word_id = word2id[start_word]

    with torch.no_grad():
        for _ in range(n_words):
            h, scores = step(one_hot(word_id), h)
            word_id = scores.argmax().item()      # best guess becomes next input
            result.append(id2word[word_id])

    return " ".join(result)


print("generated:", generate("the", 8))
print("original :", sentence)


generated: the quick brown fox jumps over the lazy dog
original : the quick brown fox jumps over the lazy dog


## 10. Look at the memory

Print `h` after each word. Notice the two `the` rows are different vectors —
that difference is what lets the RNN predict `quick` the first time and `lazy` the second.


In [14]:
h = torch.zeros(hidden_size)

with torch.no_grad():
    print(f"{'after word':>10}   memory h")
    for word_id in inputs:
        h, _ = step(one_hot(word_id), h)
        vals = "  ".join(f"{v:+.2f}" for v in h)
        print(f"{id2word[word_id]:>10}   {vals}")


after word   memory h
       the   -0.87  -0.49  +0.88  +0.44  +0.42  +0.88
     quick   +0.95  -0.91  -0.14  +0.63  -0.97  +0.96
     brown   +0.90  +0.79  -0.99  +0.95  +0.93  -0.21
       fox   +0.32  +0.86  +0.98  +0.99  +0.84  -0.99
     jumps   -0.97  -0.98  -0.17  +0.98  -0.94  -0.87
      over   -0.93  +0.10  -0.86  -0.77  +1.00  +0.96
       the   +0.31  +0.89  +1.00  -0.83  -0.65  +0.97
      lazy   -0.63  -0.97  -0.95  -0.91  -0.92  +0.89
